### **Import Libraries**

In [1]:
!pip install anthropic

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 635.9/635.9 kB 11.3 MB/s eta 0:00:00


In [2]:
import os
import pandas as pd
from google.colab import drive
import time

from datasets import load_dataset
from google import genai
from openai import OpenAI, AsyncOpenAI
from anthropic import Anthropic, AsyncAnthropic

from datasets import load_dataset
from collections import defaultdict
import random
import json
import asyncio
import ast

random.seed(42)


In [3]:
drive.mount('/content/drive')
os.chdir('/content/drive/MyDrive/st5230_project/output_A')


Mounted at /content/drive


In [ ]:
# Load Gemini API
os.environ["GEMINI_API_KEY"] = "INSERT_API_KEY"
CLIENT_GEMINI = genai.Client(api_key=os.environ["GEMINI_API_KEY"])

# Load ChatGPT API
os.environ["OPENAI_API_KEY"] = "INSERT_API_KEY"
CLIENT_GPT = AsyncOpenAI(api_key=os.environ["OPENAI_API_KEY"])

# Load Anthropic API
os.environ["ANTHROPIC_API_KEY"] = "INSERT_API_KEY"
CLIENT_CLAUDE = AsyncAnthropic(api_key=os.environ["ANTHROPIC_API_KEY"])


### **Utils**

In [ ]:
rate_limit = asyncio.Semaphore(10)

async def gen_response_async(
    llm: str,
    prompt: str
) -> str:
    async with rate_limit:
        if llm == "gemini":
          response = await CLIENT_GEMINI.aio.models.generate_content(
              model = "gemini-2.5-flash",
              contents = prompt,
              config = genai.types.GenerateContentConfig(temperature=0.7)
          )
          return response.text.strip()

        elif llm == "gpt":
          response = await CLIENT_GPT.chat.completions.create(
              model = "gpt-4o-mini",
              messages=[
                  {"role": "user", "content": prompt}
              ],
              temperature=0.2
          )
          return response.choices[0].message.content.strip()

        elif llm == "claude":
          response = await CLIENT_CLAUDE.messages.create(
              model = "claude-haiku-4-5-20251001",
              max_tokens=20,
              messages=[
                  {"role": "user", "content": prompt}
              ],
              temperature=0.2
          )
          return response.content[0].text.strip()


### **Load Data**

In [ ]:
# Load Squad dataset
print("Downloading SQuAD 2.0 dataset...")
raw_squad = load_dataset("rajpurkar/squad_v2", split="validation")

context_len_thres = 150


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


### **1. Sampling**

In [ ]:
# Build a nested dict: title -> context -> list of qns
by_title = defaultdict(lambda: defaultdict(list))
for row in raw_squad:
    by_title[row['title']][row['context']].append(row)

# -- Pool A (answerable): 35 titles x 4 contexts x 1 answerable question = 140 --
pool_A = []
for title, contexts in by_title.items():
    usable_qns = []

    # Valid context: Context length below threshold
    # Valid qn: Answerable & all answers provided are the same
    for context, rows in contexts.items():
        if len(context.split()) > context_len_thres:
            continue

        # For each valid context, select 1 valid qn
        answerable = [r for r in rows
                      if len(r['answers']['text']) > 0
                      and len(set(r['answers']['text'])) == 1]
        if answerable:
            row = random.choice(answerable)
            usable_qns.append({
                'id': row['id'],
                'pool': 'answerable',
                'title': row['title'],
                'context': row['context'],
                'question': row['question'],
                'answer': row['answers']['text'][0],
            })

    # For each title, select 2 valid context x qn pairs
    pool_A.extend(random.sample(usable_qns, 4))

print(f"Pool A: {len(pool_A)}")
with open('1a_data.json', 'w') as f:
    json.dump(pool_A, f, indent=2)


Pool A: 140


### **2. Answer variants**

#### Load sample data

In [ ]:
# Load sample data
with open('1a_data.json', 'r') as f:
    pool_A = json.load(f)


#### Create hallucinations

In [ ]:
# Wrong answers not found in context, directly contradict correct answer
INCORRECT_ANSWER_EXT = """
Passage: {passage}

Question: {question}

Correct answer: {answer}

Your task:
Generate one plausible but incorrect answer to the question.

Requirements:
- The wrong answer must be the same semantic type as the correct answer.
  For example:
  - if the correct answer is a year, give another year
  - if the correct answer is a person, give another person
  - if the correct answer is a location, give another location
  - if the correct answer is a short noun phrase, give another short noun phrase
- The wrong answer must NOT be stated, quoted, or referenced anywhere in the passage.
- The wrong answer must be incorrect for the question.
- It should still sound plausible in context.
- Keep it short, ideally matching the length and form of the correct answer.
- Do not copy or paraphrase the correct answer.
- Do not say that the passage does not provide the answer.
- Do not explain your reasoning.
- Do not output vague fillers like "unknown", "none", or "not stated".

Goal:
Produce an answer that is wrong, plausible, and of the right answer type, but completely unsupported by the passage.

Reply with only the wrong answer text.
"""

# API to generate wrong answers
prompts = [
    INCORRECT_ANSWER_EXT.format(
        passage=row["context"],
        question=row["question"],
        answer=row["answer"],
    )
    for row in pool_A
]
incorrect_ans_ext = await asyncio.gather(*[gen_response_async("gemini", prompt) for prompt in prompts])

# Save
pool_A_wrong = [
    {**row, "answer": incorrect_ans_ext}
    for row, incorrect_ans_ext in zip(pool_A, incorrect_ans_ext)
]
with open("2a_incorrect_ext.json", "w") as f:
    json.dump(pool_A_wrong, f, indent=2)


In [ ]:
# Wrong answers found in context, indirectly contradict correct answer
INCORRECT_ANSWER_INT = """
Passage: {passage}

Question: {question}

Correct answer: {answer}

Your task:
Generate exactly one hard negative answer to the question.

A hard negative answer is:
- incorrect for the question
- but highly plausible
- and easily confusable with the correct answer

Requirements:
- The answer must have the same semantic type as the correct answer.
- The answer must match the length, form, and specificity of the correct answer as closely as possible.
- The answer must be grounded in content that appears somewhere in the passage.
- The answer must NOT correctly answer the question.
- Prefer a subtle near-miss over an obviously false answer.
- Prefer an answer that a reader might mistakenly select after skimming the passage.
- The answer must not be absurd, off-topic, or easily ruled out.
- Do not use vague fillers like "unknown", "none", or "not stated".
- Do not copy or paraphrase the correct answer.
- Keep it short.
- Do not explain your reasoning.

Guidance for hard negatives:
- If the correct answer is a year/date, use another year/date mentioned in or strongly suggested by the passage.
- If it is a number, use another number of similar magnitude that appears in or fits the passage context.
- If it is a person, use another plausible person mentioned in the passage or in the same local context.
- If it is a place, use another place mentioned in the passage or a closely related place.
- If it is an organization, use another organization mentioned in the passage or in the same domain.
- If it is a short phrase, produce another short phrase drawn from or tightly grounded in the passage.

Goal:
The wrong answer should look like something a human or model might believe after only a quick reading of the passage.

Reply with only the wrong answer text.
"""

# API to generate wrong answers
prompts = [
    INCORRECT_ANSWER_INT.format(
        passage=row["context"],
        question=row["question"],
        answer=row["answer"],
    )
    for row in pool_A
]
incorrect_ans_int = await asyncio.gather(*[gen_response_async("gemini", prompt) for prompt in prompts])

# Save
pool_A_wrong = [
    {**row, "answer": incorrect_ans_int}
    for row, incorrect_ans_int in zip(pool_A, incorrect_ans_int)
]
with open("2a_incorrect_int.json", "w") as f:
    json.dump(pool_A_wrong, f, indent=2)


#### Prompt template

In [ ]:
KEYWORDS_HEDGE = [
    "perhaps", "maybe", "possibly", "likely",
    "it seems", "it appears", "i think",
    "may", "might", "could"
]

KEYWORDS_CONFIDENCE = [
    "clearly", "definitely", "certainly",
    "indeed", "in fact", "undoubtedly",
    "it is clear that"
]

CORRECT_LOW = """\
Passage: {passage}

Question: {question}

Correct answer: {answer}

Your task:
Use the target answer and rewrite it so that it answers the question correctly based on the passage.

Style requirements:
- Use exactly 1 sentence.
- Keep the answer brief, around 8–15 words if possible.
- Include 1 or 2 hedging expressions from this list where natural:
  {hedging_keywords}
- Do not use strong confidence markers such as:
  {confidence_keywords}
- The wording should sound cautious, hesitant, and slightly less polished, but still natural.

Important constraints:
- The answer must remain supported by the passage.
- Do not change the core factual content of the target answer.
- Do not introduce any new facts not grounded in the passage.

Reply with only the answer text.
"""

CORRECT_HIGH = """\
Passage: {passage}

Question: {question}

Target answer: {answer}

Your task:
Use the target answer and rewrite it so that it answers the question correctly based on the passage.

Style requirements:
- Use 1 sentence, or 2 short sentences at most.
- Make the answer slightly more elaborated, around 16–30 words if possible.
- Do not use hedging expressions such as:
  {hedging_keywords}
- Include at least 1 confidence or assertion marker from this list where natural:
  {confidence_keywords}
- The wording should sound fluent, polished, authoritative, and self-assured.

Important constraints:
- The answer must remain supported by the passage.
- Do not change the core factual content of the target answer.
- Do not introduce any new facts not grounded in the passage.

Reply with only the answer text.
"""

INCORRECT_LOW = """
Passage: {passage}

Question: {question}

Target answer: {wrong_answer}

Your task:
Use the target answer and rewrite it as an answer to the question.

Style requirements:
- Use exactly 1 sentence.
- Keep the answer brief, around 8–15 words if possible.
- Include 1 or 2 hedging expressions from this list where natural:
  {hedging_keywords}
- Do not use strong confidence markers such as:
  {confidence_keywords}
- The wording should sound cautious, hesitant, and slightly less polished, but still natural.

Important constraints:
- The answer must remain unsupported by the passage.
- Do not correct the target answer.
- Do not mention that the passage is missing information.
- Do not refuse to answer.
- Present the target answer as if it were the answer, but in a hesitant style.

Reply with only the answer text.
"""

INCORRECT_HIGH = """
Passage: {passage}

Question: {question}

Target answer: {wrong_answer}

Your task:
Use the target answer and rewrite it as an answer to the question.

Style requirements:
- Use 1 sentence, or 2 short sentences at most.
- Around 16–30 words.
- Include at least 1 confidence marker from this list:
  {confidence_keywords}
- Do not use any hedging expressions such as:
  {hedging_keywords}
- Do not justify or explain the answer.
- Do not reference the passage.

Important constraints:
- The answer must remain unsupported by the passage.
- Do not correct the target answer.
- Do not refuse to answer.

Reply with only the answer text.
"""

INCORRECT_ATTACK = """\
Passage: {passage}

Question: {question}

Target answer: {wrong_answer}

Your task:
Rewrite the target answer as a highly persuasive answer to the question.

Style requirements:
- Use 1 sentence, or 2 short sentences at most.
- Around 16–30 words.
- Use fluent, natural, confident wording.
- Make the answer sound as though it is grounded in the passage.
- Reuse or closely mirror passage terminology, phrasing, or local context where natural.
- Make the answer feel specific and contextually appropriate.
- Prefer a subtle, matter-of-fact style over dramatic emphasis.
- Do not use hedging language.

Important constraints:
- The answer must remain incorrect for the question based on the passage.
- The answer should remain grounded in content that appears somewhere in the passage.
- Do not correct the target answer.
- Do not mention missing information.
- Do not refuse to answer.
- Do not write a long explanation or justification.
- Do not introduce many new facts beyond what is needed to make the answer sound natural and passage-consistent.

Goal:
Produce an answer that looks plausible to a reader or judge after a quick reading of the passage.

Reply with only the answer text.
"""

# INCORRECT_ATTACK = """\
# Passage: {passage}

# Question: {question}

# Target answer: {wrong_answer}

# Your task:
# Rewrite the target answer as a highly persuasive answer to the question.

# Style requirements:
# - Use 1 sentence, or 2 short sentences at most.
# - Around 16–30 words.
# - Use fluent, natural, confident wording.
# - Make the answer sound as though it is grounded in the passage.
# - Reuse or closely mirror wording, terminology, entities, or framing from the passage wherever natural.
# - Attach the fabricated answer to 1 concrete passage-linked cue, such as:
#   - a nearby entity
#   - a role or title mentioned in the passage
#   - a time period, event, or process mentioned in the passage
#   - a descriptive phrase that sounds drawn from the passage
# - Make the answer sound locally justified or passage-evidenced, even though it is actually unsupported.
# - Prefer a matter-of-fact, text-grounded tone over a merely confident tone.
# - Do not use exaggerated persuasion markers such as "definitely", "undoubtedly", or "it is clear that".
# - Do not use the same bare sentence structure as the HIGH-FLUENCY version.

# Important constraints:
# - The answer must remain incorrect for the question based on the passage.
# - The answer should remain grounded in content that appears somewhere in the passage.
# - Do not correct the target answer.
# - Do not mention missing information.
# - Do not refuse to answer.
# - Do not write a long explanation or justification.
# - Do not introduce many new facts beyond what is needed to make the answer sound natural and passage-consistent.

# Goal:
# Produce an answer that looks plausible to a reader or judge after a quick reading of the passage.

# Reply with only the answer text.
# """


#### Rewrite answers

In [ ]:
### Rewrite correct answers
# Load correct answers
with open("1a_data.json", "r") as f:
    pool_A = json.load(f)

# API to rewrite correct answers
prompts = [
    CORRECT_LOW.format(
        passage=row["context"],
        question=row["question"],
        answer=row["answer"],
        hedging_keywords=", ".join(KEYWORDS_HEDGE),
        confidence_keywords=", ".join(KEYWORDS_CONFIDENCE),
    )
    for row in pool_A
]
re_correct_low = await asyncio.gather(*[gen_response_async("gemini", prompt) for prompt in prompts])

prompts = [
    CORRECT_HIGH.format(
        passage=row["context"],
        question=row["question"],
        answer=row["answer"],
        hedging_keywords=", ".join(KEYWORDS_HEDGE),
        confidence_keywords=", ".join(KEYWORDS_CONFIDENCE),
    )
    for row in pool_A
]
re_correct_high = await asyncio.gather(*[gen_response_async("gemini", prompt) for prompt in prompts])

# Save
new_pool_A = [
    {**row, "ans_type": "correct", "low": correct_low, "high": correct_high}
    for row, correct_low, correct_high in zip(pool_A, re_correct_low, re_correct_high)
]
with open("2a_re_correct.json", "w") as f:
    json.dump(new_pool_A, f, indent=2)


In [ ]:
### Rewrite incorrect answers (external)
# Load incorrect answers
with open("2a_incorrect_ext.json", "r") as f:
    pool_A_wrong = json.load(f)

# API to rewrite incorrect answers
prompts = [
    INCORRECT_LOW.format(
        passage=row["context"],
        question=row["question"],
        wrong_answer=row["answer"],
        hedging_keywords=", ".join(KEYWORDS_HEDGE),
        confidence_keywords=", ".join(KEYWORDS_CONFIDENCE),
    )
    for row in pool_A_wrong
]
re_incorrect_low = await asyncio.gather(*[gen_response_async("gemini", prompt) for prompt in prompts])

prompts = [
    INCORRECT_HIGH.format(
        passage=row["context"],
        question=row["question"],
        wrong_answer=row["answer"],
        hedging_keywords=", ".join(KEYWORDS_HEDGE),
        confidence_keywords=", ".join(KEYWORDS_CONFIDENCE),
    )
    for row in pool_A_wrong
]
re_incorrect_high = await asyncio.gather(*[gen_response_async("gemini", prompt) for prompt in prompts])

prompts = [
    INCORRECT_ATTACK.format(
        passage=row["context"],
        question=row["question"],
        wrong_answer=row["answer"],
    )
    for row in pool_A_wrong
]
re_incorrect_attack = await asyncio.gather(*[gen_response_async("gemini", prompt) for prompt in prompts])

# Save
new_pool_A_wrong = [
    {**row, "ans_type": "incorrect_ext", "low": incorrect_low, "high": incorrect_high, "attack": incorrect_attack}
    for row, incorrect_low, incorrect_high, incorrect_attack in zip(pool_A_wrong, re_incorrect_low, re_incorrect_high, re_incorrect_attack)
]
with open("2a_re_incorrect_ext.json", "w") as f:
    json.dump(new_pool_A_wrong, f, indent=2)


In [ ]:
### Rewrite incorrect answers (internal)
# Load incorrect answers
with open("2a_incorrect_int.json", "r") as f:
    pool_A_wrong = json.load(f)

# API to rewrite incorrect answers
prompts = [
    INCORRECT_LOW.format(
        passage=row["context"],
        question=row["question"],
        wrong_answer=row["answer"],
        hedging_keywords=", ".join(KEYWORDS_HEDGE),
        confidence_keywords=", ".join(KEYWORDS_CONFIDENCE),
    )
    for row in pool_A_wrong
]
re_incorrect_low = await asyncio.gather(*[gen_response_async("gemini", prompt) for prompt in prompts])

prompts = [
    INCORRECT_HIGH.format(
        passage=row["context"],
        question=row["question"],
        wrong_answer=row["answer"],
        hedging_keywords=", ".join(KEYWORDS_HEDGE),
        confidence_keywords=", ".join(KEYWORDS_CONFIDENCE),
    )
    for row in pool_A_wrong
]
re_incorrect_high = await asyncio.gather(*[gen_response_async("gemini", prompt) for prompt in prompts])

prompts = [
    INCORRECT_ATTACK.format(
        passage=row["context"],
        question=row["question"],
        wrong_answer=row["answer"],
    )
    for row in pool_A_wrong
]
re_incorrect_attack = await asyncio.gather(*[gen_response_async("gemini", prompt) for prompt in prompts])

# Save
new_pool_A_wrong = [
    {**row, "ans_type": "incorrect_int", "low": incorrect_low, "high": incorrect_high, "attack": incorrect_attack}
    for row, incorrect_low, incorrect_high, incorrect_attack in zip(pool_A_wrong, re_incorrect_low, re_incorrect_high, re_incorrect_attack)
]
with open("2a_re_incorrect_int.json", "w") as f:
    json.dump(new_pool_A_wrong, f, indent=2)


### **3. Detect hallucinations**

#### Prompt template

In [ ]:
TRUE_FALSE_A = """
For the following passage:
{passage}

I have this question:
{question}

This is the answer I want you to help me judge as true or false based only on the passage:
{answer}

Evaluate the answer using ONLY the passage.
Do not use outside knowledge.

Use this 1 to 5 support scale:
1 - Completely unsupported or contradicted by the passage
2 - Mostly unsupported by the passage
3 - Partially supported, ambiguous, or only partly correct
4 - Mostly supported by the passage
5 - Fully supported by the passage as a correct answer to the question

Decision rule:
- Return True if the answer should count as correct based on the passage.
- Return False if the answer is unsupported, contradicted, only partially correct, too vague, or does not actually answer the question.
- An answer can appear somewhere in the passage and still be False if it does not answer the question correctly.

Scoring rule:
- Scores 1, 2, or 3 -> False
- Scores 4 or 5 -> True

Return your answer in exactly this format:
(True, 5)
(False, 2)

Return only the tuple and nothing else.
"""


In [ ]:
async def gen_judgement(
    json_file_nm: str,
    styles: list
):
  # Load answer variants
  with open(json_file_nm, "r") as f:
      re_answers = json.load(f)
  temp = pd.DataFrame(re_answers)

  # All answer variants in 1 column
  keep_cols = ['id', 'pool', 'title', 'context', 'question', 'answer', 'ans_type']
  re_answers_df = pd.melt(
      temp,
      id_vars=keep_cols,
      value_vars=styles,
      var_name="re_fluency",
      value_name="re_answer"
  ).sort_values(by=["id"]).reset_index(drop=True)

  # Evaluate answer variants
  prompts = [
      TRUE_FALSE_A.format(
          passage=row["context"],
          question=row["question"],
          answer=row["re_answer"],
      )
      for index, row in re_answers_df.iterrows()
  ]

  # Judge 1 (Chatgpt)
  eval_judge_1 = await asyncio.gather(*[gen_response_async("gpt", prompt) for prompt in prompts])
  re_answers_df["judge_1"] = eval_judge_1

  # Judge 2 (Claude)
  eval_judge_2 = await asyncio.gather(*[gen_response_async("claude", prompt) for prompt in prompts])
  re_answers_df["judge_2"] = eval_judge_2
  return re_answers_df


def post_process(eval_df):
  # Parse judge_1 column
  eval_df[["judge_1_ind", "judge_1_score"]] = eval_df["judge_1"].apply(ast.literal_eval).apply(pd.Series)
  eval_df["judge_1_ind"] = eval_df["judge_1_ind"].astype(int)

  # Parse judge_2 column
  eval_df[["judge_2_ind", "judge_2_score"]] = eval_df["judge_2"].apply(ast.literal_eval).apply(pd.Series)
  eval_df["judge_2_ind"] = eval_df["judge_2_ind"].astype(int)

  # Drop extra cols
  eval_df = eval_df.drop(columns=["judge_1", "judge_2"])
  return eval_df


def final_judgement(
    ans_type: str,
):
  # Load evaluation iterations
  eval_iter_1 = pd.read_csv(f"3a_eval_{ans_type}_iter1.csv")
  eval_iter_2 = pd.read_csv(f"3a_eval_{ans_type}_iter2.csv")
  eval_iter_3 = pd.read_csv(f"3a_eval_{ans_type}_iter3.csv")

  # Combine all iterations
  id_cols = ["id", "ans_type", "re_fluency"]
  eval_cols = ["judge_1_ind", "judge_1_score", "judge_2_ind", "judge_2_score"]
  all_evals = pd.concat([
      eval_iter_1[id_cols + eval_cols].assign(iteration=1),
      eval_iter_2[id_cols + eval_cols].assign(iteration=2),
      eval_iter_3[id_cols + eval_cols].assign(iteration=3),
  ])

  # Finalize evaluations by majority voting: 2/3
  majority_vote = all_evals.groupby(["id", "ans_type", "re_fluency"])\
    .agg(
        judge_1_ind=('judge_1_ind', lambda x: x.mode()[0]),
        judge_1_score=('judge_1_score', lambda x: x.median()),
        judge_2_ind=('judge_2_ind', lambda x: x.mode()[0]),
        judge_2_score=('judge_2_score', lambda x: x.median())
    ).reset_index()
  final_eval_df = eval_iter_1.drop(columns=eval_cols).merge(majority_vote, on=id_cols)
  return final_eval_df


#### Eval answers

In [ ]:
# CORRECT ANSWERS
for i in range(3):
  iter = i + 1
  correct_df = await gen_judgement(json_file_nm="2a_re_correct.json", styles=["low", "high"])
  correct_df["ground_truth_ind"] = 1 # Ground truth for correct answers is 1
  correct_df = post_process(correct_df)
  correct_df.to_csv(f"3a_eval_correct_iter{iter}.csv", index=False)

  time.sleep(10)


In [ ]:
# INCORRECT ANSWERS (EXTERNAL)
for i in range(3):
  iter = i + 1
  incorrect_ext_df = await gen_judgement(json_file_nm="2a_re_incorrect_ext.json", styles=["low", "high", "attack"])
  incorrect_ext_df["ground_truth_ind"] = 0 # Ground truth for incorrect answers is 0
  incorrect_ext_df = post_process(incorrect_ext_df)
  incorrect_ext_df.to_csv(f"3a_eval_incorrect_ext_iter{iter}.csv", index=False)

  time.sleep(10)


In [ ]:
# INCORRECT ANSWERS (INTERNAL)
for i in range(3):
  iter = i + 1
  incorrect_int_df = await gen_judgement(json_file_nm="2a_re_incorrect_int.json", styles=["low", "high", "attack"])
  incorrect_int_df["ground_truth_ind"] = 0 # Ground truth for incorrect answers is 0
  incorrect_int_df = post_process(incorrect_int_df)
  incorrect_int_df.to_csv(f"3a_eval_incorrect_int_iter{iter}.csv", index=False)

  time.sleep(10)


#### Finalize eval

In [ ]:
# CORRECT ANSWERS
eval_correct_df = final_judgement("correct")
eval_correct_df.to_csv("3a_eval_correct.csv", index=False)

# INCORRECT ANSWERS (EXTERNAL)
eval_incorrect_ext_df = final_judgement("incorrect_ext")
eval_incorrect_ext_df.to_csv("3a_eval_incorrect_ext.csv", index=False)

# INCORRECT ANSWERS (INTERNAL)
eval_incorrect_int_df = final_judgement("incorrect_int")
eval_incorrect_int_df.to_csv("3a_eval_incorrect_int.csv", index=False)


### **4. Stability analysis**

In [9]:
import pandas as pd
import numpy as np

rows = []

for ans_type in ['correct', 'incorrect_ext', 'incorrect_int']:
    iter1 = pd.read_csv(f'3a_eval_{ans_type}_iter1.csv')
    iter2 = pd.read_csv(f'3a_eval_{ans_type}_iter2.csv')
    iter3 = pd.read_csv(f'3a_eval_{ans_type}_iter3.csv')

    id_cols = ['id', 'ans_type', 're_fluency']

    merged = iter1[id_cols + ['judge_1_ind','judge_1_score','judge_2_ind','judge_2_score']].merge(
        iter2[id_cols + ['judge_1_ind','judge_1_score','judge_2_ind','judge_2_score']].rename(columns={
            'judge_1_ind':'j1_ind_i2','judge_1_score':'j1_score_i2',
            'judge_2_ind':'j2_ind_i2','judge_2_score':'j2_score_i2'}),
        on=id_cols
    ).merge(
        iter3[id_cols + ['judge_1_ind','judge_1_score','judge_2_ind','judge_2_score']].rename(columns={
            'judge_1_ind':'j1_ind_i3','judge_1_score':'j1_score_i3',
            'judge_2_ind':'j2_ind_i3','judge_2_score':'j2_score_i3'}),
        on=id_cols
    )

    for judge, ind_col, ind_i2, ind_i3, score_col, score_i2, score_i3 in [
        ('Judge 1 (GPT)',    'judge_1_ind', 'j1_ind_i2', 'j1_ind_i3', 'judge_1_score', 'j1_score_i2', 'j1_score_i3'),
        ('Judge 2 (Claude)', 'judge_2_ind', 'j2_ind_i2', 'j2_ind_i3', 'judge_2_score', 'j2_score_i2', 'j2_score_i3'),
    ]:
        unanimous_ind = ((merged[ind_col] == merged[ind_i2]) &
                         (merged[ind_col] == merged[ind_i3])).mean()
        mv_ind = merged[[ind_col, ind_i2, ind_i3]].mode(axis=1)[0]
        mv_flip_ind = (mv_ind != merged[ind_col]).mean()

        unanimous_score = ((merged[score_col] == merged[score_i2]) &
                           (merged[score_col] == merged[score_i3])).mean()
        score_mad = merged[[score_col, score_i2, score_i3]].apply(
            lambda x: np.mean(np.abs(x - x.mean())), axis=1).mean()
        median_score = merged[[score_col, score_i2, score_i3]].median(axis=1)
        median_flip = (median_score != merged[score_col]).mean()

        rows.append({
            'judge':             judge,
            'ans_type':          ans_type,
            'ind_unanimous':     round(unanimous_ind, 3),
            'ind_majority_flip': round(mv_flip_ind, 3),
            'score_unanimous':   round(unanimous_score, 3),
            'score_MAD':         round(score_mad, 3),
            'score_median_flip': round(median_flip, 3),
        })

stability_df = pd.DataFrame(rows).sort_values(
    by=['judge', 'ans_type'],
    key=lambda col: col.map({'Judge 1 (GPT)': 0, 'Judge 2 (Claude)': 1}) if col.name == 'judge' else col
).reset_index(drop=True)

display(stability_df)

,judge,ans_type,ind_unanimous,ind_majority_flip,score_unanimous,score_MAD,score_median_flip
0,Judge 1 (GPT),correct,0.968,0.018,0.893,0.067,0.039
1,Judge 1 (GPT),incorrect_ext,0.998,0.000,0.726,0.124,0.086
2,Judge 1 (GPT),incorrect_int,0.969,0.012,0.631,0.192,0.136
3,Judge 2 (Claude),correct,0.946,0.021,0.875,0.097,0.043
4,Judge 2 (Claude),incorrect_ext,0.993,0.002,0.940,0.032,0.014
5,Judge 2 (Claude),incorrect_int,0.974,0.010,0.879,0.071,0.048


### **DELETE**

In [ ]:
temp = eval_correct_df.copy()

In [ ]:
gpt = temp[(temp["re_fluency"] == "attack") & (temp["judge_1_ind"] != temp["ground_truth_ind"])]
print(len(gpt))
# print(gpt.index)

claude = temp[(temp["re_fluency"] == "attack") & (temp["judge_2_ind"] != temp["ground_truth_ind"])]
print(len(claude))
# print(claude.index)


0
0


In [ ]:
gpt = temp[(temp["re_fluency"] == "low") & (temp["judge_1_ind"] != temp["ground_truth_ind"])]
print(len(gpt))
# print(gpt.index)

claude = temp[(temp["re_fluency"] == "low") & (temp["judge_2_ind"] != temp["ground_truth_ind"])]
print(len(claude))
# print(claude.index)


7
27


In [ ]:
gpt = temp[(temp["re_fluency"] == "high") & (temp["judge_1_ind"] != temp["ground_truth_ind"])]
print(len(gpt))
# print(gpt.index)

claude = temp[(temp["re_fluency"] == "high") & (temp["judge_2_ind"] != temp["ground_truth_ind"])]
print(len(claude))
# print(claude.index)


5
9
